This is another core retrieval concept that separates basic RAG from production systems.

Let’s break it down in the same structured way:

👉 What

👉 Why

👉 Analogy

👉 Types

👉 Where it fits

🧠 1. What are Pre-Filtering & Post-Filtering?

🔍 Simple Definition

Type	           When it Happens	            Purpose

Pre-filtering	  Before retrieval	       Narrow search space

Post-filtering	  After retrieval	          Improve result quality

🔄 Pipeline View

User Query

   ↓

Pre-filtering 🔒

   ↓

Retrieval 🔍

   ↓

Post-filtering 🧹

   ↓

Final Context → LLM

🔵 2. Pre-Filtering

🔍 What is it?

👉 Restricting which documents are eligible before search happens

❗ Problem it Solves

Without pre-filtering:

Query → Search entire database → Noise

👉 Too many irrelevant results

👉 Slower retrieval

👉 Higher cost

✅ Solution

Apply constraints like:

domain

category

time

source

metadata

🧠 Analogy (Amazon Shopping)

You want to buy a phone.

👉 Without filtering:

You see TVs, laptops, accessories, etc.

👉 With pre-filter:

Category = Mobile Phones

Brand = Apple

Now search becomes focused.

👉 That’s pre-filtering

🧪 Example

User query:

"How to reduce investment risk?"

Pre-filter:

{
  "domain": "finance"
}

👉 Only finance documents are searched

🔥 Types of Pre-Filtering

Metadata filtering

    domain = healthcare

    year = 2024

LLM-based filter extraction

    LLM decides filters from query

Access control filtering

    user permissions

Time-based filtering

   recent documents only

🚀 Impact

Faster retrieval

Higher precision

Lower cost

More controllable system

❌ Risk

👉 Over-filtering

Too strict → Missing relevant docs

🟢 3. Post-Filtering

🔍 What is it?

👉 Filtering results after retrieval

❗ Problem it Solves

Even after retrieval:

Top-K results ≠ fully relevant

👉 Some results are:

partially relevant

noisy

misleading

✅ Solution

Evaluate retrieved docs and remove bad ones

🧠 Analogy (Job Recruitment)

Step 1:

👉 Resume screening (retrieval)

Step 2:

👉 Interview evaluation (post-filter)

Even if resumes match keywords, candidates may not fit.

👉 That’s post-filtering

🧪 Example

Retrieved docs:

1. Diversification reduces investment risk ✔

2. Stock markets fluctuate ❌ (not directly relevant)

3. Risk management strategies ✔

Post-filter:

Keep → 1, 3

Remove → 2

🔥 Types of Post-Filtering

LLM relevance filtering

YES/NO decision

Score-based filtering

relevance score threshold

Rule-based filtering

keyword checks

Semantic similarity filtering

cosine threshold

🚀 Impact

Improves final answer quality

Removes noise

Improves grounding

❌ Trade-off

Adds latency (LLM calls)

Adds cost

🟣 4. Deep Comparison

Aspect	   Pre-filtering	          Post-filtering

Stage	      Before retrieval	      After retrieval

Goal	      Reduce search space	      Improve quality

Speed	      Faster	                  Slower

Risk	      Missing data	          Extra cost

Control	High (rules/metadata)	  High (LLM reasoning)

🧠 5. Combined Analogy (Best Way to Understand)

🎓 University Admission Process

Step 1: Pre-filter

👉 Only students with:

70% marks

Science background

(Reject others immediately)

Step 2: Evaluation (Retrieval)

👉 Shortlist candidates

Step 3: Post-filter

👉 Interview + skill test

(Select best candidates)

👉 Final selection = Pre + Post filtering

🔥 6. Where It Fits in Full RAG

User Query

   ↓

Intent Validation

   ↓

Query Formulation

   ↓

Query Expansion

   ↓

Pre-filtering 🔒

   ↓

Hybrid Retrieval 🔍

   ↓

Post-filtering 🧹

   ↓

Reranking ⭐

   ↓

Final Context

   ↓

LLM Answer

🧠 7. Deep Insight (Very Important)

👉 These two solve different failure modes

Problem	Solution

Too many irrelevant docs	Pre-filtering

Retrieved docs not precise	Post-filtering

In [1]:
import os
import json
import chromadb
from sentence_transformers import SentenceTransformer
from langchain_openai import ChatOpenAI
from dotenv import load_dotenv

load_dotenv()

# OpenAI Chat Model
llm = ChatOpenAI(model="gpt-4o-mini", temperature=0)

# Embedding model
embedding_model = SentenceTransformer("all-MiniLM-L6-v2")

# Chroma client
client = chromadb.Client()
collection = client.create_collection(name="multi_domain_rag")

In [2]:
# ==========================================
# 2. SAMPLE DOCUMENTS (GENERIC DOMAINS)
# ==========================================
documents = [
    "Diabetes is a chronic disease that affects blood sugar levels.",
    "Hypertension increases the risk of heart disease and stroke.",
    "Stock markets fluctuate based on economic conditions.",
    "Diversification reduces investment risk in finance.",
    "Artificial Intelligence enables machines to learn from data.",
    "Cloud computing provides scalable infrastructure over the internet.",
    "Neural networks are used in deep learning applications."
]

metadatas = [
    {"domain": "healthcare", "type": "disease"},
    {"domain": "healthcare", "type": "risk"},
    {"domain": "finance", "type": "market"},
    {"domain": "finance", "type": "investment"},
    {"domain": "technology", "type": "ai"},
    {"domain": "technology", "type": "cloud"},
    {"domain": "technology", "type": "ml"}
]

# Add to Chroma
embeddings = [embedding_model.encode(doc).tolist() for doc in documents]

collection.add(
    documents=documents,
    embeddings=embeddings,
    metadatas=metadatas,
    ids=[f"id_{i}" for i in range(len(documents))]
)

In [9]:
# ==========================================
# 3. FILTER VALIDATION + BUILDING
# ==========================================
VALID_DOMAINS = ["healthcare", "finance", "technology"]
VALID_TYPES = ["disease", "risk", "market", "investment", "ai", "cloud", "ml"]

def clean_filters(filters):
    cleaned = {}

    if filters.get("domain") in VALID_DOMAINS:
        cleaned["domain"] = filters["domain"]

    if filters.get("type") in VALID_TYPES:
        cleaned["type"] = filters["type"]

    return cleaned


def build_chroma_filter(filters):
    conditions = []

    for k, v in filters.items():
        if v:
            conditions.append({k: v})

    if not conditions:
        return None

    if len(conditions) == 1:
        return conditions[0]

    return {"$and": conditions}

In [10]:
# ==========================================
# 4. PRE-FILTERING (LLM)
# ==========================================
def detect_filters_llm(query):
    prompt = f"""
    Extract metadata filters from query.

    Allowed values:
    domain: healthcare, finance, technology
    type: disease, risk, market, investment, ai, cloud, ml

    Return ONLY JSON:
    {{
      "domain": "...",
      "type": "..."
    }}

    Query: {query}
    """

    response = llm.invoke(prompt)

    try:
        filters = json.loads(response.content)
        return filters
    except:
        return {}

In [11]:
# ==========================================
# 5. POST-FILTERING (LLM)
# ==========================================
def post_filter_docs(query, docs):
    filtered_docs = []

    for doc in docs:
        prompt = f"""
        Check if this document is relevant.

        Query: {query}
        Document: {doc}

        Answer YES or NO only.
        """

        response = llm.invoke(prompt)

        if "YES" in response.content.upper():
            filtered_docs.append(doc)

    return filtered_docs

In [12]:
# ==========================================
# 6. RETRIEVAL (FIXED)
# ==========================================
def hybrid_retrieval(query):

    # Step 1: LLM filter extraction
    raw_filters = detect_filters_llm(query)
    print("🧠 Raw filters:", raw_filters)

    # Step 2: Clean filters
    cleaned_filters = clean_filters(raw_filters)
    print("🧹 Cleaned filters:", cleaned_filters)

    # Step 3: Convert to Chroma format
    chroma_filter = build_chroma_filter(cleaned_filters)
    print("🔎 Chroma filter:", chroma_filter)

    # Step 4: Vector search
    query_embedding = embedding_model.encode(query).tolist()

    results = collection.query(
        query_embeddings=[query_embedding],
        n_results=10,
        where=chroma_filter
    )

    docs = results["documents"][0]
    print("📄 Retrieved:", docs)

    # Step 5: Post-filter
    refined_docs = post_filter_docs(query, docs)
    print("✅ After post-filter:", refined_docs)

    return refined_docs[:5]

In [13]:
# ==========================================
# 7. GENERATION
# ==========================================
def generate_answer(query, context_docs):
    context = "\n".join(context_docs)

    prompt = f"""
    Answer ONLY using the context.

    Context:
    {context}

    Question: {query}
    """

    response = llm.invoke(prompt)
    return response.content

In [14]:
# ==========================================
# 8. FULL PIPELINE
# ==========================================
def rag_pipeline(query):

    context_docs = hybrid_retrieval(query)

    if not context_docs:
        return {"error": "No relevant documents found"}

    answer = generate_answer(query, context_docs)

    return {
        "query": query,
        "context": context_docs,
        "answer": answer
    }

In [15]:
# ==========================================
# 9. TEST
# ==========================================
if __name__ == "__main__":

    queries = [
        "What is diabetes?",
        "How to reduce financial risk?",
        "Explain neural networks"
    ]

    for q in queries:
        print("\n============================")
        print("Query:", q)

        result = rag_pipeline(q)

        print("\n🎯 FINAL OUTPUT:")
        print(json.dumps(result, indent=2))


Query: What is diabetes?
🧠 Raw filters: {}
🧹 Cleaned filters: {}
🔎 Chroma filter: None
📄 Retrieved: ['Diabetes is a chronic disease that affects blood sugar levels.', 'Hypertension increases the risk of heart disease and stroke.', 'Diversification reduces investment risk in finance.', 'Cloud computing provides scalable infrastructure over the internet.', 'Artificial Intelligence enables machines to learn from data.', 'Neural networks are used in deep learning applications.', 'Stock markets fluctuate based on economic conditions.']
✅ After post-filter: ['Diabetes is a chronic disease that affects blood sugar levels.']

🎯 FINAL OUTPUT:
{
  "query": "What is diabetes?",
  "context": [
    "Diabetes is a chronic disease that affects blood sugar levels."
  ],
  "answer": "Diabetes is a chronic disease that affects blood sugar levels."
}

Query: How to reduce financial risk?
🧠 Raw filters: {'domain': 'finance', 'type': 'risk'}
🧹 Cleaned filters: {'domain': 'finance', 'type': 'risk'}
🔎 Chrom